In [1]:
# HW10
# December 5, 2025
# Madelyn Carlson

# Import packages
import os
import scipy.io as sio
from self_py_fun.HW10Fun import *
from sklearn.linear_model import LogisticRegression as LR
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# In HW7, you have the chance to visualize a truncated EEG dataset stratified by
# target and non-target stimulus type.
#
# The fundamental problem of P300 ERP-BCI speller system is to perform a binary classification.
#
# In HW10, you are asked to implement the binary classification using various methods,
# and evaluate the model performance with a testing dataset.
#
# You will use K114_001_BCI_TRN_Truncated_Data_0.5_6.mat as a training set, and
# K114_001_BCI_FRT_Truncated_Data_0.5_6.mat as a testing set.
#
# Notice that here, we do not split training/testing within K114_001_BCI_TRN_Truncated_Data_0.5_6.mat
# because each row is not entirely independent of each other due to the special structure of the dataset.

# Global constants:
np.random.seed(100)
bp_low = 0.5
bp_upp = 6
electrode_num = 16
# Change the following directory to your own one.
parent_dir = '/Users/madelyncarlson/Documents/GitHub/BIOS-584'
parent_data_dir = '{}/data'.format(parent_dir)
time_index = np.linspace(0, 800, 25)
electrode_name_ls = ['F3', 'Fz', 'F4', 'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CP4', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
subject_name = 'K114'
# create a new folder called K114
subject_dir = '{}/{}'.format(parent_dir, subject_name)
if not os.path.exists(subject_dir):
    os.mkdir(subject_dir)

char_trn = 'THE0QUICK0BROWN0FOX'
char_trn_size = len(char_trn)

# Step 1: Import dataset
# Step 1.1: TRN dataset
trn_data_name = '{}_001_BCI_TRN_Truncated_Data_{}_{}'.format(subject_name, bp_low, bp_upp)
trn_data_dir = '{}/{}.mat'.format(parent_data_dir, trn_data_name)
eeg_trn_obj = sio.loadmat(trn_data_dir)

# eeg_trn_obj is a dictionary!
print(eeg_trn_obj.keys())
eeg_trn_signal = eeg_trn_obj['Signal']
print(eeg_trn_signal.shape) # 3420, 400
eeg_trn_type = eeg_trn_obj['Type']
print(eeg_trn_type.shape) # 3420, 1
eeg_trn_type = np.squeeze(eeg_trn_type, axis=1)

# Step 1.2: FRT dataset
# The following code should be completed by students themselves.
# you should be able to obtain relevant data files named
# eeg_frt_signal and eeg_frt_type
# Write your own code below:
frt_data_name = '{}_001_BCI_FRT_Truncated_Data_{}_{}'.format(subject_name, bp_low, bp_upp)
frt_data_dir = '{}/{}.mat'.format(parent_data_dir, frt_data_name)
eeg_frt_obj = sio.loadmat(frt_data_dir)
print(eeg_frt_obj.keys())
eeg_frt_signal = eeg_frt_obj['Signal']
print(eeg_frt_signal.shape) # (1296, 400)
eeg_frt_type = eeg_frt_obj['Type']
print(eeg_frt_type.shape) # (1296, 1)
eeg_frt_type = np.squeeze(eeg_frt_type, axis=1)

# You have completed the exploratory data analysis in HW7 and HW8.
# The dataset has been carefully reviewed by Dr. Jane E. Huggins,
# so we do not need to worry about missing, outliers, errors of the dataset.

# Step 2: Fit classification models
# You will try the following methods:
# Logistic Regression,
# Linear Discriminant Analysis,
# Support Vector Machine (sometimes called support vector classification)
# You do not need to modify the parameters of each classifier
# except for LogisticRegression: set max_iter=1000
# Write your own code below:
# first, scale data
scaler_obj = StandardScaler()
eeg_trn_signal_scaled = scaler_obj.fit_transform(eeg_trn_signal)
eeg_frt_signal_scaled = scaler_obj.transform(eeg_frt_signal)

# instantiate models
logistic_obj = LR(max_iter=1000) # logistic regression
lda_obj = LDA() # LDA
svm_obj = SVC(probability=True) # SVM

# train models
logistic_obj.fit(eeg_trn_signal_scaled, eeg_trn_type) # logistic regression
lda_obj.fit(eeg_trn_signal_scaled, eeg_trn_type) # LDA
svm_obj.fit(eeg_trn_signal_scaled, eeg_trn_type) # SVM

# predict (FRT data)
log_reg_preds = logistic_obj.predict(eeg_frt_signal_scaled)# logistic regression
lda_preds = lda_obj.predict(eeg_frt_signal_scaled) # LDA
svm_preds = svm_obj.predict(eeg_frt_signal_scaled) # SVM

y_preds_three_method = np.stack([log_reg_preds, lda_preds, svm_preds], axis=0).T
print(y_preds_three_method[:5, :])

log_reg_probs = logistic_obj.predict_proba(eeg_frt_signal_scaled)
print(log_reg_probs[:5, :])
print()

lda_probs = lda_obj.predict_proba(eeg_frt_signal_scaled)
print(lda_probs[:5, :])
print()

svm_probs = svm_obj.predict_proba(eeg_frt_signal_scaled)
print(svm_probs[:5, :])
print()

# Step 3: Evaluate model performance on both TRN and FRT files
# Step 3.1: Prediction accuracy on TRN files
# We will compute the probability of each stimulus with .predict_proba()
# before we convert the stimulus-level probability to character-level probability.
# You are asked to generate stimulus-level probability for each method on TRN files,
# denoted as logistic_y_trn, lda_y_trn, and svm_y_trn.
# Write your own code below:
# TRN
log_reg_trn_preds = logistic_obj.predict(eeg_trn_signal_scaled)
lda_trn_preds = lda_obj.predict(eeg_trn_signal_scaled)
svm_trn_preds = svm_obj.predict(eeg_trn_signal_scaled)

model_preds_trn = {
    "Logistic Regression (TRN)": log_reg_trn_preds,
    "Linear Discriminant Analysis (TRN)": lda_trn_preds,
    "Support Vector Machine (TRN)": svm_trn_preds
}
for model, preds in model_preds_trn.items():
    print(f"{model} Results (TRN):")
    print(classification_report(eeg_trn_type, preds))
    print("\n")

# using predict_proba
logistic_y_trn = logistic_obj.predict_proba(eeg_trn_signal_scaled)
print("Logistic regression:", logistic_y_trn.shape)
print()

lda_y_trn = lda_obj.predict_proba(eeg_trn_signal_scaled)
print("LDA:", lda_y_trn.shape)
print()

svm_y_trn = svm_obj.predict_proba(eeg_trn_signal_scaled)
print("SVM:", svm_y_trn.shape)
print()

logistic_y_trn_class1 = logistic_y_trn[:, 1]
print("Logistic regression:", logistic_y_trn_class1[:5])
print()

lda_y_trn_class1 = lda_y_trn[:, 1]
print("LDA:", lda_y_trn_class1[:5])

svc_y_trn_class1 = svm_y_trn[:, 1]
print("SVM:", svc_y_trn_class1[:5])
print()

# FRT
model_preds = {
    "Logistic Regression": log_reg_preds,
    "Linear Discriminant Analysis": lda_preds,
    "Support Vector Machine": svm_preds
}
for model, preds in model_preds.items():
    print(f"{model} Results:")
    print(classification_report(eeg_frt_type, preds))
    print("\n")

# Step 3.2: Prediction accuracy on FRT files
# Similarly, you are asked to generate stimulus-level probability for each method on FRT files,
# denoted as logistic_y_frt, lda_y_frt, and svm_y_frt.
# Write your own code below:
logistic_y_frt = logistic_obj.predict_proba(eeg_frt_signal_scaled)
print("Logistic regression shape:", logistic_y_frt.shape)
print()

lda_y_frt = lda_obj.predict_proba(eeg_frt_signal_scaled)
print("LDA shape:", lda_y_frt.shape)
print()

svm_y_frt = svm_obj.predict_proba(eeg_frt_signal_scaled)
print("SVM shape:", svm_y_frt.shape)
print()

logistic_y_frt_class1 = logistic_y_frt[:, 1]
print("Logistic regression:", logistic_y_frt_class1[:5])
print() # observations: slightly lower than LDA. Moderate confidence.

lda_y_frt_class1 = lda_y_frt[:, 1]
print("LDA:", lda_y_frt_class1[:5])
print() # observations: more confident

svm_y_frt_class1 = svm_y_frt[:, 1]
print("SVM:", svm_y_frt_class1[:5])
print() # observations: high precision but low recall

dict_keys(['__header__', '__version__', '__globals__', 'Code', 'IndexBegin', 'IndexTag', 'LetterTable', 'Signal', 'Text', 'Type'])
(3420, 400)
(3420, 1)
dict_keys(['__header__', '__version__', '__globals__', 'BandLow', 'BandUpp', 'Code', 'IndexBegin', 'IndexTag', 'LetterTable', 'Signal', 'Text', 'Type'])
(1296, 400)
(1296, 1)
[[-1 -1 -1]
 [-1 -1 -1]
 [ 1  1  1]
 [-1 -1 -1]
 [-1 -1 -1]]
[[7.98360081e-01 2.01639919e-01]
 [9.84616656e-01 1.53833444e-02]
 [4.61013947e-04 9.99538986e-01]
 [9.66673577e-01 3.33264232e-02]
 [9.98608747e-01 1.39125336e-03]]

[[9.09048248e-01 9.09517524e-02]
 [9.96181940e-01 3.81805999e-03]
 [3.36463987e-04 9.99663536e-01]
 [9.98689799e-01 1.31020068e-03]
 [9.99836578e-01 1.63421951e-04]]

[[0.58432716 0.41567284]
 [0.96481431 0.03518569]
 [0.0145064  0.9854936 ]
 [0.98248258 0.01751742]
 [0.98840731 0.01159269]]

Logistic Regression (TRN) Results (TRN):
              precision    recall  f1-score   support

          -1       0.96      0.98      0.97      2850


In [3]:
# Step 4: Convert binary classification probability to character-level accuracy
# This involves advanced data manipulation, so you do not need to write any new code.
# Please run the following code to view the final results.
eeg_trn_code = eeg_trn_obj['Code']
eeg_frt_code = eeg_frt_obj['Code']
char_frt = convert_raw_char_to_alphanumeric_stype(eeg_frt_obj['Text'])
# raw format is different from the current 6x6 layout characters.
char_frt_size = len(char_frt)
frt_seq_size = int(eeg_frt_signal.shape[0]/char_frt_size/12)

# Logistic regression
print('Logistic Regression on TRN:')
logistic_letter_mat_trn, logistic_letter_prob_mat_trn = streamline_predict(
    logistic_y_trn, eeg_trn_type, eeg_trn_code, char_trn_size, trn_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(logistic_letter_mat_trn)
print(list(char_trn)) # This is the true spelling characters for training set!
logistic_trn_accuracy = np.mean(logistic_letter_mat_trn == np.array(list(char_trn))[:, np.newaxis], axis=0)

print('Logistic Regression on FRT:')
logistic_letter_mat_frt, logistic_letter_prob_mat_frt = streamline_predict(
    logistic_y_frt, eeg_frt_type, eeg_frt_code, char_frt_size, frt_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(logistic_letter_mat_frt)
print(list(char_frt)) # This is the true spelling characters for testing set!
logistic_frt_accuracy = np.mean(logistic_letter_mat_frt == np.array(list(char_frt))[:, np.newaxis], axis=0)

# LDA:
print('LDA on TRN:')
lda_letter_mat_trn, lda_letter_prob_mat_trn = streamline_predict(
    lda_y_trn, eeg_trn_type, eeg_trn_code, char_trn_size, trn_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(lda_letter_mat_trn)
print(list(char_trn)) # This is the true spelling characters for training set!
lda_trn_accuracy = np.mean(lda_letter_mat_trn == np.array(list(char_trn))[:, np.newaxis], axis=0)

print('LDA on FRT:')
lda_letter_mat_frt, lda_letter_prob_mat_frt = streamline_predict(
    lda_y_frt, eeg_frt_type, eeg_frt_code, char_frt_size, frt_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(lda_letter_mat_frt)
print(list(char_frt)) # This is the true spelling characters for testing set!
lda_frt_accuracy = np.mean(lda_letter_mat_frt == np.array(list(char_frt))[:, np.newaxis], axis=0)

# SVM:
print('Support Vector Machine on TRN:')
svm_letter_mat_trn, svm_letter_prob_mat_trn = streamline_predict(
    svm_y_trn, eeg_trn_type, eeg_trn_code, char_trn_size, trn_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(svm_letter_mat_trn)
print(list(char_trn)) # This is the true spelling characters for training set!
svm_trn_accuracy = np.mean(svm_letter_mat_trn == np.array(list(char_trn))[:, np.newaxis], axis=0)

print('Support Vector Machine on FRT:')
svm_letter_mat_frt, svm_letter_prob_mat_frt = streamline_predict(
    svm_y_frt, eeg_frt_type, eeg_frt_code, char_frt_size, frt_seq_size,
    stimulus_group_set, eeg_rcp_array
)
print(svm_letter_mat_frt)
print(list(char_frt)) # This is the true spelling characters for training set!
svm_frt_accuracy = np.mean(svm_letter_mat_frt == np.array(list(char_frt))[:, np.newaxis], axis=0)

Logistic Regression on TRN:
[['Z' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T' 'T']
 ['H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H']
 ['E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E' 'E']
 ['0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0']
 ['Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q' 'Q']
 ['U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U' 'U']
 ['C' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C' 'C']
 ['K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K' 'K']
 ['0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0']
 ['B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B' 'B']
 ['R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R' 'R']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W' 'W']
 ['N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N']
 ['4' '0' '0' '0' '0' '0' '

In [5]:
print(f" logistic_trn_accuracy: {logistic_trn_accuracy}")
print(f" lda_trn_accuracy: {lda_trn_accuracy}")
print(f" svm_trn_accuracy: {svm_trn_accuracy}")

print(f" logistic_frt_accuracy: {logistic_frt_accuracy}")
print(f" lda_frt_accuracy: {lda_frt_accuracy}")
print(f" svm_frt_accuracy: {svm_frt_accuracy}")

 logistic_trn_accuracy: [0.84210526 1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.        ]
 lda_trn_accuracy: [0.78947368 0.94736842 1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.        ]
 svm_trn_accuracy: [0.94736842 1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.        ]
 logistic_frt_accuracy: [0.66666667 0.92592593 0.92592593 1.        ]
 lda_frt_accuracy: [0.7037037  0.85185185 0.92592593 0.96296296]
 svm_frt_accuracy: [0.66666667 0.88888889 0.88888889 1.        ]
